# ALS avec implicit

La bibliothèque `implicit` entraîne un modèle ALS sur les interactions MovieLens. Les notes d'au moins 4/5 sont traitées comme des favoris. L'application fournit uniquement les IDs TMDB de ses favoris.

L'entraînement et la comparaison sont dans `src`. Le notebook lit le rapport et le modèle produits par cette commande, lancée depuis la racine du dépôt :

```powershell
.\.venv\Scripts\python.exe -m src.training.train
```

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src/models/collaborative.py").is_file()
    and (path / "config/evaluation.json").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dépôt movies-and-me-recommender.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.exploration import load_movielens
from src.data.id_mapper import IdMapper

# Les barres de progression Jupyter sont facultatives.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="IProgress not found.*")
    from src.models.collaborative import ImplicitRecommender

## Comparaison sur la validation

Les trois modèles sont évalués sur les mêmes utilisateurs et le même catalogue, avec une limite de dix recommandations. Les favoris viennent de l'entraînement ; les films à retrouver viennent de la période suivante.

Recall@10 mesure la part des films attendus présents dans les dix recommandations. NDCG@10 tient aussi compte de leur position : retrouver un film en début de liste rapporte davantage.

In [2]:
report_path = PROJECT_ROOT / "reports/als_comparison.json"
if not report_path.is_file():
    raise FileNotFoundError(
        "Rapport absent. Lancer d'abord : python -m src.training.train"
    )
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"Catalogue : {report['catalog']['movies']} films")
print(f"Validation : {report['coverage']['eligible_users']} utilisateurs MovieLens, "
      f"{report['coverage']['eligible_positive_targets']} films attendus dans le catalogue")
print("Essai ALS retenu :", report["selection"]["selected_trial"])

comparison = pd.DataFrame({
    name: result["metrics"] for name, result in report["models"].items()
}).T.rename(index={"popularity": "Popularité", "genres": "Genres", "als": "ALS"})
comparison = comparison.rename(columns={
    "recall_at_10": "Recall@10", "ndcg_at_10": "NDCG@10",
})
comparison["Recall@10"] = comparison["Recall@10"].mul(100)
display(comparison.rename(columns={"Recall@10": "Recall@10 (%)"}).round(4))

Catalogue : 5223 films
Validation : 20 utilisateurs MovieLens, 333 films attendus dans le catalogue
Essai ALS retenu : als_64


,Recall@10 (%),NDCG@10
Popularité,4.5726,0.0389
Genres,0.0000,0.0000
ALS,4.8142,0.0623


In [3]:
trials = pd.DataFrame([
    {"essai": trial["name"], **trial["config"], **trial["evaluation"]["metrics"]}
    for trial in report["trials"]
])
trials["recall_at_10"] = trials["recall_at_10"].mul(100)
display(trials.set_index("essai").rename(columns={
    "recall_at_10": "Recall@10 (%)", "ndcg_at_10": "NDCG@10",
}).round(4))
print("Choix : meilleur NDCG@10, puis Recall@10 en cas d'égalité.")

,factors,regularization,alpha,iterations,random_state,num_threads,Recall@10 (%),NDCG@10
essai,,,,,,,,
als_32,32,0.1,20.0,30,42,1,5.6931,0.0471
als_64,64,0.1,20.0,30,42,1,4.8142,0.0623
als_32_regularized,32,1.0,20.0,30,42,1,5.3565,0.0443


Choix : meilleur NDCG@10, puis Recall@10 en cas d'égalité.


La validation affichée porte sur 20 utilisateurs MovieLens. L'essai à 64 facteurs obtient le meilleur NDCG@10 (0,0623), tandis que celui à 32 facteurs a le meilleur Recall@10 (5,69 %). Le choix repose d'abord sur le NDCG@10.

Cet échantillon est trop petit pour conclure sur les favoris de l'application. Le jeu de test final n'intervient pas dans cette sélection.

## Recommandations à partir des favoris

Le modèle sélectionné est rechargé depuis le disque. Les titres viennent de MovieLens, sans requête à TMDB.

In [4]:
model_path = PROJECT_ROOT / "artifacts/implicit-als"
if not model_path.is_dir():
    raise FileNotFoundError(
        "Modèle absent. Lancer d'abord : python -m src.training.train"
    )
recommender = ImplicitRecommender.load(model_path)
assert recommender.metadata["selection"] == report["selection"], \
    "Le modèle et le rapport ne correspondent pas : essai choisi différent."
assert recommender.metadata["sources"] == report["sources"], \
    "Le modèle et le rapport ne correspondent pas : données différentes."
assert recommender.metadata["validation_metrics"] == report["models"]["als"]["metrics"], \
    "Le modèle et le rapport ne correspondent pas : scores différents."
tables = load_movielens(PROJECT_ROOT / "data/raw")
mapper = IdMapper(tables["links"])
titles = tables["movies"].set_index("movieId")["title"].to_dict()

def title_for(tmdb_id):
    return titles.get(mapper.to_movie_id(tmdb_id), "Titre absent de MovieLens")

def show_recommendations(result):
    frame = pd.DataFrame(result["recommendations"], columns=["tmdb_id", "score"])
    frame.insert(1, "titre", frame["tmdb_id"].map(title_for))
    print("Classement :", result["method"])
    display(frame)

shape = recommender.training_data.matrix.shape
print(f"Entraînement : {shape[0]} utilisateurs MovieLens et {shape[1]} films.")
user_factors_before = recommender.model.user_factors.copy()
item_factors_before = recommender.model.item_factors.copy()

Entraînement : 521 utilisateurs MovieLens et 5223 films.


In [5]:
favorites = [27205, 157336, 155, 603]
print("Favoris :")
display(pd.DataFrame({
    "tmdb_id": favorites, "titre": [title_for(identifier) for identifier in favorites],
}))
result = recommender.recommend_with_method(favorites, n=10)
show_recommendations(result)
assert not set(favorites).intersection(
    row["tmdb_id"] for row in result["recommendations"]
)

Favoris :


,tmdb_id,titre
0,27205,Inception (2010)
1,157336,Interstellar (2014)
2,155,"Dark Knight, The (2008)"
3,603,"Matrix, The (1999)"


Classement : als


,tmdb_id,titre,score
0,120,"Lord of the Rings: The Fellowship of the Ring,...",0.533886
1,550,Fight Club (1999),0.502725
2,278,"Shawshank Redemption, The (1994)",0.498596
3,122,"Lord of the Rings: The Return of the King, The...",0.487304
4,77,Memento (2000),0.469171
5,14160,Up (2009),0.468960
6,49026,"Dark Knight Rises, The (2012)",0.456802
7,16869,Inglourious Basterds (2009),0.456546
8,194,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",0.434950
9,857,Saving Private Ryan (1998),0.431125


Les scores ALS servent au classement. Ils ne sont ni des notes sur 5, ni des probabilités. Seuls les favoris transmis sont exclus des propositions.

Le second exemple utilise Toy Story, Finding Nemo et The Incredibles. Le profil est calculé à partir de cette liste de favoris, sans identifiant utilisateur MovieLens.

In [6]:
animation_favorites = [862, 12, 9806]
print("Favoris :", ", ".join(title_for(identifier) for identifier in animation_favorites))
show_recommendations(recommender.recommend_with_method(animation_favorites, n=10))

Favoris : Toy Story (1995), Finding Nemo (2003), Incredibles, The (2004)
Classement : als


,tmdb_id,titre,score
0,585,"Monsters, Inc. (2001)",0.501754
1,808,Shrek (2001),0.419548
2,863,Toy Story 2 (1999),0.410197
3,809,Shrek 2 (2004),0.378305
4,22,Pirates of the Caribbean: The Curse of the Bla...,0.367367
5,671,Harry Potter and the Sorcerer's Stone (a.k.a. ...,0.323575
6,121,"Lord of the Rings: The Two Towers, The (2002)",0.310305
7,122,"Lord of the Rings: The Return of the King, The...",0.299245
8,2062,Ratatouille (2007),0.298572
9,425,Ice Age (2002),0.262066


## Sans favori connu

Les favoris absents du catalogue d'entraînement ne contribuent pas au profil. Si aucun favori n'est connu, le classement utilise la popularité : le score est le nombre d'utilisateurs MovieLens ayant aimé le film pendant l'entraînement.

In [7]:
empty_profile = recommender.recommend_with_method([], n=5)
unknown_profile = recommender.recommend_with_method([987654321], n=5)
print("Liste vide :", empty_profile["method"])
print("Favori inconnu :", unknown_profile["method"])
assert empty_profile == unknown_profile
show_recommendations(empty_profile)

Liste vide : popularity
Favori inconnu : popularity
Classement : popularity


,tmdb_id,titre,score
0,278,"Shawshank Redemption, The (1994)",218.0
1,680,Pulp Fiction (1994),208.0
2,13,Forrest Gump (1994),203.0
3,274,"Silence of the Lambs, The (1991)",192.0
4,603,"Matrix, The (1999)",176.0


## Sauvegarde et rechargement

Les assertions vérifient que les facteurs du modèle restent inchangés après plusieurs appels et qu'une sauvegarde rechargée conserve les films recommandés et leurs scores. La copie est écrite dans un dossier temporaire, supprimé à la fin de la cellule.

In [8]:
expected = result["recommendations"]
after_other_profiles = recommender.recommend(favorites, n=10)
assert [row["tmdb_id"] for row in after_other_profiles] == [row["tmdb_id"] for row in expected]
np.testing.assert_allclose(
    [row["score"] for row in after_other_profiles], [row["score"] for row in expected],
    rtol=1e-6, atol=1e-7,
)
np.testing.assert_array_equal(recommender.model.user_factors, user_factors_before)
np.testing.assert_array_equal(recommender.model.item_factors, item_factors_before)

with TemporaryDirectory(prefix="movies-als-") as temporary:
    copy_path = Path(temporary) / "model"
    recommender.save(copy_path)
    restored = ImplicitRecommender.load(copy_path)
    actual = restored.recommend(favorites, n=10)
    assert [row["tmdb_id"] for row in actual] == [row["tmdb_id"] for row in expected]
    np.testing.assert_allclose(
        [row["score"] for row in actual], [row["score"] for row in expected],
        rtol=1e-6, atol=1e-7,
    )

print("Facteurs inchangés après les appels.")
print("Mêmes films et mêmes scores après sauvegarde et rechargement.")

Facteurs inchangés après les appels.
Mêmes films et mêmes scores après sauvegarde et rechargement.
